In [ ]:
# %load_ext nb_mypy
# %nb_mypy Off

In [ ]:
from __future__ import annotations
import numpy as np
import random
import copy
import importlib
import matplotlib.pyplot as plt

from typing import Tuple, List
from numpy import array, zeros

# from Big_Class import Big_Class  # already imported one NETfuncs is imported
from User_Variables import User_Variables  # already imported one NETfuncs is imported
from Network_Structure import Network_Structure  # already imported one NETfuncs is imported
from Big_Class import Big_Class
from Network_State import Network_State
from Networkx_Net import Networkx_Net
from Color_Scheme import Color_Scheme
import matrix_functions, functions, statistics, plot_funcs, solve, plot_funcs, colors

## colors

In [ ]:
importlib.reload(colors)

colors_lst, red, cmap = colors.color_scheme()
cmap

Colorscheme = Color_Scheme(show=True)

# Set up Network

In [ ]:
## Parameters - can't be confused from runs

## task type
# task_type='Iris_classification'
task_type='Regression'

## input dataset type
dataset_type = 'uniform_random'  # dataset sampled from uniform random distribution at each iteration
# dataset_type = 'alternating ones'  # dataset is one at a single input and the rest zero, alternating between single inputs

## network type
# net_type='square'
net_type='FC'
# net_type='FC_connected_outputs'
# net_type='beads'
if net_type == 'beads':
    R_update = 'beads'

# for square network
# net_height=8
# net_length=7
# rand_seed=36  # seed for random nodes as inputs, outputs, ground
net_height=16
net_length=16

## specify # of nodes
Nin: int = 2
Nout: int = 2
if task_type == 'Iris_classification':
    Nin = 4
    Nout = 3
Ninter: int = 0
    
in_nodes: np.ndarray(int) = array([], dtype=np.int_)
out_nodes: np.ndarray(int) = array([], dtype=np.int_)

# measure accuracy every # steps
measure_accuracy_every = 15

supress_prints: bool = True  # whether to print information during training or not
# supress_prints: bool = False  # whether to print information during training or not
# include_Power: bool = True
include_Power: bool = False

# ## Networkx sizes
scale: float = 50.0
squish: float = 0.01
# scale: float = 1.0
# squish: float = 1.0
    
## use a node with p=0 during all steps or not
add_ground=True
# add_ground=False

R_max: float = 2.0
R_min: float = 0.02
    
## print time, measured and desired outputs every print_every iterations
print_every = 1

## calculate cosine similarity between gradient descent and my scheme
# calculate_cosine_sim = True
calculate_cosine_sim = False

In [ ]:
## Parameters - stuff that you can confuse in different runs

## task matrix X
# M_values: np.ndarray = array([0.15, 0.2, 0.025, 0.1, 0.02, 0.3, 0.35, 0.15, 0.03, 0.25, 0.1, 0.15, 0.02, 0.3, 0.35, 0.15, 0.03])
# M_values: np.ndarray = array([0.85, 0.65, 0.25, 0.1, 0.02, 0.03, 0.35, 0.15, 0.03, 0.25, 0.1, 0.15, 0.02, 0.3, 0.35, 0.15, 0.03])
# M_values: np.ndarray = array([0.05, 0.25, 0.1, 0.15, 0.02, 0.3])  # for p-inv task 3x2
# M_values: np.ndarray = array([0.7, 0.15, 0.2, 0.35])
M_values: np.ndarray = array([2/4, 1/4, 0.1, 0.35, 0.75, 0.04])
# M_values: np.ndarray = array([0.2, 0.2, 0.3, 0.25, 0.35, 0.25])
# M_values: np.ndarray = array([0.1, 0.1, 0.1, 0.2])
# M_values: np.ndarray = array([0.1, 0.2, 0.3, 0.4, 0.1, 0.1])
# M_values: np.ndarray = array([])

## Whether to normalize the task matrix so that the sum over a line is 0.75
# normalize_M = True
normalize_M = False

normalize = 0.75  # normalize lines in task matrix

## random M task generation
random_state_M = 42

## seed for random nodes as inputs, outputs, ground in square network
rand_seed=35  

## shuffle of training set
random_state = 52

# hysteresis = 0.2
hysteresis = 0.0
    
## method to update resistances - physical property of the system
# R_update: str = 'R_propto_dp'
# R_update: str = 'R_propto_sqrt_dp'
R_update: str = 'deltaR_propto_dp'
# R_update: str = 'deltaR_propto_dp_decay'
# R_update: str = 'deltaR_propto_dp_nonlin'
# R_update: str = 'deltaR_propto_dp_nonlin_decay'
# R_update: str = 'grad_desc'
# R_update: str = 'R_propto_Q'
# R_update: str = 'deltaR_propto_Q'
# R_update: str = 'deltaR_propto_Power'
# R_update: str = 'R_propto_Power'
# R_update: str = 'R_propto_Q_exp'

if R_update in ['deltaR_propto_dp_nonlin', 'deltaR_propto_dp_nonlin_decay'] or hysteresis:
    normalize_loss = True
else:
    normalize_loss = False
    
normalize_loss = True  # temporarily force it

loss_type = 'MSE'
# loss_type = 'MAE'
    
## resistance-pressure proportionality factor
gamma = np.array([1.0])

## use 1 or 2 sampled pressures at every time step
# use_p_tag: bool = True  
use_p_tag: bool = False

## how many loop iterations to stay under the same sampled p
# stay_sample: int = 2 
# stay_sample: int = 1
stay_sample: int = 1

## Use constant step size regardless of loss
# normalize_step = True
normalize_step = False

In [ ]:
factor = 1

# anneal = False    
anneal = True
T_annealing: float = 2/factor  # for cosine annealing scheme - larger time means slower decay
# T_annealing: float = 8/factor  # for cosine annealing scheme - larger time means slower decay

# length of training dataset
iterations = int(1800*stay_sample*factor)  # number of sampled of p

decay_R = 2e-3

## learning rate
alpha: float = 0.17  # for network combine attempt
if task_type == 'Iris_classification':
    if R_update == 'R_propto_dp':
        alpha = 0.03
    elif R_update == 'deltaR_propto_dp':
        alpha = 0.05
    elif R_update == 'deltaR_propto_Q':
        alpha = 0.05
    elif R_update == 'deltaR_propto_Power':
        alpha = 0.25
alpha_vec = np.array([alpha])
# alpha_vec = np.arange(2.5, 10.7, 0.2)
print('alphas ', alpha_vec)
    
# training_scheme: str = 'multip'
training_scheme: str = 'Adaline'
# training_scheme: str = 'GD_like'

R_vec_i = np.ones(Nin*Nout + Nin + Nout)
# R_vec_i = array([0.2968,    0.1333,    0.0594,    1.0000,    0.1484,    0.0667,    0.0247,   0.0377])  # Rs that solve task
# R_vec_i = array([ 0.12975,  0.30035,  0.4444 ,  0.615  ,  0.02225,  0.19285, 0.1165 , -0.0541 ])
# R_vec_i = array([0.1290,    0.4000,    0.0258,    1.0000,    0.3225,    0.0625,    0.0095,    0.0328])
# R_vec_i = 2*array([1.9, 1., 0.85, 1.05, 0.9, 1., 0.85, 1.05, 0.9, 3., 0.85, 1.05, 0.9, 1., 0.85])
# R_vec_i = np.ones(Nin*Nout + Nout) + np.random.normal(loc=0.0, scale=0.1, size=Nin*Nout + Nout)
# R_in_t_pseudo = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.8/30Aug_R_pseudo_vs_network/R_in_t_pseudo.npy')
# R_vec_i = R_in_t_pseudo[0]
# # Replace negative values with 1
# R_vec_i = np.where(R_vec_i < 0, np.mean(R_vec_i), R_vec_i)

In [ ]:
## User Variables - Keep those since not in use Apr2025
access_interNodes: bool = False
noise_to_extra: bool = False  # add noise to extra outputs 

In [ ]:
import User_Variables
importlib.reload(User_Variables)
from User_Variables import User_Variables

## Variables class - mostly user choices
Variabs = User_Variables(iterations,\
                         Nin, \
                         Nout, \
                         gamma, \
                         R_update, \
                         training_scheme, \
                         use_p_tag, \
                         normalize_loss, \
                         supress_prints, \
                         task_type, \
                         dataset_type, \
                         measure_accuracy_every, \
                         normalize_step = normalize_step, \
                         hysteresis = hysteresis, \
                         R_max = R_max, \
                         R_min = R_min, \
                         anneal = anneal, \
                         T_annealing = T_annealing, \
                         Ninter = Ninter, \
                         decay_R = decay_R)
Variabs.assign_alpha_vec(alpha)

# create random entries for M
if np.size(M_values) < Nin*Nout:
    M_values = functions.random_gen_M(random_state_M, Nin*Nout) 
    
if normalize_M:
    ## Deal with normalizing task matrix M so task is concave
    M_values_norm = functions.normalize_M(M_values, normalize, Nin, Nout)
    Variabs.create_dataset_and_targets(random_state=random_state, M_values=M_values_norm, train_size = 0.8)
else:
    Variabs.create_dataset_and_targets(random_state=random_state, M_values=M_values, train_size = 0.8)
Variabs.create_noise_for_extras()

## Assign input and output nodes a.f.o lattice size and row choice
inOutGround_tuple = matrix_functions.build_input_output_and_ground(Variabs.Nin, Variabs.Nout, in_nodes=in_nodes, Ninter=Ninter,
                                                                   out_nodes=out_nodes, add_ground=add_ground, 
                                                                   net_type=net_type, seed=rand_seed, net_height=net_height, 
                                                                   net_len=net_length)

## prints
print('input_nodes_arr ', inOutGround_tuple[0])
print('extraInput_nodes_arr ', inOutGround_tuple[1])
print('inter_nodes_arr ', inOutGround_tuple[2])
print('output_nodes_arr ', inOutGround_tuple[3])
print('extraOutput_nodes_arr ', inOutGround_tuple[4])
print('ground_nodes_arr ', inOutGround_tuple[5])

In [ ]:
import Network_Structure
importlib.reload(Network_Structure)
from Network_Structure import Network_Structure
importlib.reload(matrix_functions)

## Big Class containing all classes in Network Simulation
BigClass = Big_Class(Variabs)

## Save color scheme in Big Class
BigClass.add_Colors(Colorscheme)

## Structure class - build incidence matrices and 1d arrays of edges
Strctr = Network_Structure(inOutGround_tuple, net_type, net_height, net_length)
# if Ninter >= 1:
#     Strctr.build_incidence('partialInter')
# else:
Strctr.build_incidence(net_type)
if training_scheme in ['GD_like', 'Adaline']:
    Strctr.build_inverse_incidence()
    if training_scheme == 'GD_like':
        Strctr.build_RM(Variabs.Nin, Variabs.Nout)
BigClass.add_Strctr(Strctr)  # add to big class

# build fictitious Network_Structure as if no inter nodes
if training_scheme == 'Adaline' and Ninter>0:
    ## Assign input and output nodes a.f.o lattice size and row choice
    inOutGround_tuple_fict = matrix_functions.build_input_output_and_ground(Variabs.Nin, Variabs.Nout, in_nodes=in_nodes,
                                                                            out_nodes=out_nodes, add_ground=add_ground, 
                                                                            net_type=net_type, seed=rand_seed,
                                                                            net_height=net_height, net_len=net_length)
    Strctr_fict = Network_Structure(inOutGround_tuple_fict, net_type, net_height, net_length)
    Strctr_fict.build_incidence(net_type)
    Strctr_fict.build_inverse_incidence()
    if training_scheme == 'GD_like':
        Strctr_fict.build_RM(Variabs.Nin, Variabs.Nout)
    BigClass.add_Strctr_fict(Strctr_fict)  # add to big class

In [ ]:
# print('U', Strctr.DM)
# print('U_dagger', Strctr.DM_dagger)
# print('grad_loss_vec', State.grad_loss_vec)

In [ ]:
import Network_State
importlib.reload(Network_State)
from Network_State import Network_State

## Initiate internal flow network state class
State = Network_State(Variabs)
if task_type == 'Iris_classification':
    State.initiate_resistances(BigClass, R_vec_i)
    State.initiate_accuracy_vec(BigClass, measure_accuracy_every)
else:
#     R_mat_i = np.concatenate((1/Variabs.M, np.ones([1, 5])), axis=0)
#     R_vec_i = R_mat_i.flatten()
    # State.initiate_resistances(BigClass, np.matmul(Strctr.DM, np.matmul(Strctr.DM_dagger, R_vec_i)))
    State.initiate_resistances(BigClass, R_vec_i)
    print(State.R_in_t[0])
BigClass.add_State(State)  # add to big class

import Networkx_Net
importlib.reload(Networkx_Net)
from Networkx_Net import Networkx_Net

## build network graphics class and plot structure
NET = Networkx_Net(scale, squish)
NET.buildNetwork(BigClass)
NET.build_pos_lattice(BigClass, plot=True, node_labels=False)
BigClass.add_NET(NET)  # add to big class

# Train

In [ ]:
GD_cost_vec = []
dK_arr = []
dK_GD_arr = []
cosine_vec = []
cosine_lin = []
cosine_nonlin = []
cosine_optimal = []
dK_step=10**(-8)

def training_loop():
    for i in range(Variabs.iterations):

        # if task is classification and iteration # is beginning of epoch
        # draw output of network as output of mean of Irises
        if i % np.shape(Variabs.X_train)[0] == 0 and task_type == 'Iris_classification':
            State.assign_targets_Iris(BigClass)
            # print('targets_mat', State.targets_mat)

        # staying stay_sample iteration under same sample
        if use_p_tag and noise_to_extra:
            k = 2*(i//stay_sample) + 1
            if not(i%4):
                k-=1
        elif use_p_tag and not(noise_to_extra):
            k = (i//stay_sample)*2 + i%2
        elif not(use_p_tag) and noise_to_extra:
            k = (i//stay_sample)
        else:
            k = (i//stay_sample)
        # print('k', k)

        for l in range(1):  # Repeat the block l times
            # # measurement modality
            # draw input and desired outputs from dataset and solve flow
            if not((i+1) % 4):  # add noise only at i=3 etc.
                State.draw_p_in_and_desired(Variabs, k, noise_to_extra=noise_to_extra)  # noise to extra nodes every 2nd iter.
                State.solve_flow_given_modality(BigClass, "measure", noise_to_extra=noise_to_extra)  # measure, don't change R's
            else:  # dont add noise to extra nodes
                State.draw_p_in_and_desired(Variabs, k)
                State.solve_flow_given_modality(BigClass, "measure")
    #         State.input_drawn_in_t.append(array([1]))
    #         State.desired_in_t.append(array([0.9, 0.1]))
    #         State.solve_flow_given_modality(BigClass, "measure")

            if calculate_cosine_sim or R_update == 'grad_desc':
                # GD cost
                # State.calc_GD_cost(Strctr, State.desired, mod='measure', func='mean_abs')
                State.calc_GD_cost(Strctr, State.desired, mod='measure', func='MSE')
                GD_cost_vec.append(State.GD_cost)

                # change in K grad desc
                # delta_K_GD = State.dK_grad_desc(Strctr, dK_step, State.desired, func='mean_abs')
                delta_K_GD = State.dK_grad_desc(Strctr, dK_step, State.desired, func='MSE')
                # print('delta_K_GD', delta_K_GD)    
                dK_GD_arr.append(delta_K_GD)

            # save R, p, u for network plots, after measurement
            if i == iterations-1:
                NET.save_R_reordered(State.R_in_t[-1], Strctr.EIEJ_plots)
                NET.save_p_reordered(State.p)
                NET.save_u_reordered(State.u, Strctr.EIEJ_plots)
#                 plot_funcs.plotNetStructure(NET.NET, BigClass, NET.pos_lattice, node_labels=False, 
#                                     R_reordered=NET.R_reordered, u_reordered=NET.u_reordered, p_reordered=NET.p_reordered)

            if include_Power:
                State.calc_Power_norm(BigClass)

            if not i % 2 and use_p_tag:  # even iterations, take another sampled pressure and measure again
                pass
            else:  # odd iterations, go to update modality and update resistances
                if net_type == 'beads':  # if beads - need a few iterations to converge
                    update_iters = 6
                else:  # if not beads, converges within a single iteration
                    update_iters = 1

                State.t += 1
                if not(State.t%print_every):  # print time, measure and desired outputs every print_every
                    # print('time=', State.t)
#                     print('measured output ', State.output)
#                     print('desired output ', State.desired)
                    # print('')
                    pass
                State.calc_loss(BigClass)
                # print('loss ', State.loss)
                loss_mean = np.mean(np.abs(State.loss), axis=1)

                if not((i + 1) % 4) and access_interNodes:
                    State.update_inter(BigClass)
                else:
                    if training_scheme in ['GD_like', 'Adaline']:
                        State.calc_update_vals_vec(BigClass)
                    State.update_input(BigClass)
                    State.update_output(BigClass)
#                     print('input_update_nxt ', State.input_update_nxt)
#                     print('output_update_nxt ', State.output_update_nxt)

                # # update modality
                for m in range(update_iters):
                    # measure and don't change resistances
                    State.solve_flow_given_modality(BigClass, "update", access_inters=access_interNodes)
                    # print('Power', np.sum(State.u**2*State.R_in_t[-1]))

                    State.update_Rs(BigClass)  # supressed - no resistance update
                        
                # # anneal learning rate
                if anneal and State.t<iterations:
                    State.update_alpha(BigClass, T_annealing)
        print('due to update_vec=', State.update_vec)
        print('resulting in resistance=', State.R_in_t[-1])
        # measure accuracy in classification task
        if i % measure_accuracy_every == 0 and task_type == 'Iris_classification' \
            and i//measure_accuracy_every<len(State.accuracy_in_t):
            if i==0:
                State.accuracy_in_t[i] = 1/3
                State.t_for_accuracy[i//measure_accuracy_every] = State.t 
            else:
                State.calculate_accuracy_testset(BigClass)
                State.accuracy_in_t[i//measure_accuracy_every] = State.accuracy 
                State.t_for_accuracy[i//measure_accuracy_every] = State.t 
#     # save R, p, u for network plots, after measurement
#     NET.save_R_reordered(State.R_in_t[-1], Strctr.EIEJ_plots)
#     NET.save_p_reordered(State.p)
#     NET.save_u_reordered(State.u, Strctr.EIEJ_plots)

    # scalar loss in time
    if task_type == 'Regression':
        denominator_dataset = Variabs.y_train
    elif task_type == 'Iris_classification':
        denominator_dataset = array([[np.mean(State.targets_mat)]])
    if loss_type == 'MAE':
        State.loss_scalar_in_t = np.mean(np.mean(np.abs(State.loss_in_t), axis=1), axis=1)
        State.loss_scalar_in_t = State.loss_scalar_in_t / np.mean(np.abs(denominator_dataset))
    elif loss_type == 'MSE':
        State.loss_scalar_in_t = np.mean(np.mean(np.square(State.loss_in_t), axis=1), axis=1)
        State.loss_scalar_in_t = State.loss_scalar_in_t / np.mean(np.square(denominator_dataset))
        # State.loss_scalar_in_t = State.loss_scalar_in_t / np.mean(np.square(denominator_dataset), axis=1)
    
#     if task_type == 'Regression':
#         if loss_type == 'MAE':
#             State.loss_norm_in_t = State.loss_in_t / np.mean(np.abs(Variabs.y_train), axis=1)[:,None,None]
#         elif loss_type == 'MSE':
#             State.loss_norm_in_t = State.loss_in_t / np.mean((Variabs.y_train)**2, axis=1)[:,None,None]
#     elif task_type == 'Iris_classification':
#         if loss_type == 'MAE':
#             State.loss_norm_in_t = State.loss_in_t / np.mean(np.mean(np.abs(Variabs.y_train), axis=1), axis=1)
#         elif loss_type == 'MSE':
#             State.loss_norm_in_t = State.loss_in_t / np.mean(np.mean((Variabs.y_train)**2, axis=1))
#     State.loss_scalar_in_t = np.mean(np.mean(np.abs(State.loss_norm_in_t), axis=1), axis=1) 

In [ ]:
import cProfile
import pstats

if __name__ == "__main__":
    profiler = cProfile.Profile()
    profiler.enable()

    if task_type == 'Regression':
        for i, alpha in enumerate(alpha_vec):
            print('alpha = ', alpha)
            Variabs.assign_alpha_vec(alpha)
            State = Network_State(Variabs)
            # R_mat_i = np.concatenate((1/Variabs.M, np.ones([1, 5])), axis=0)
            # R_vec_i = R_mat_i.flatten()
            State.initiate_resistances(BigClass, R_vec_i, add_noise=0.0)
            BigClass.add_State(State)  # add to big class
            training_loop()
            plot_funcs.plot_importants(BigClass, movmean_loss = True, include_network=False, node_labels=False)
            
    elif task_type == 'Iris_classification':
        accuracy_in_t = []
        for i in range(8):
            random_state += 1
            if random_state == 56:
                random_state += 1
            elif random_state == 61:
                random_state += 1
            print('random_state = ', random_state)
            Variabs.create_dataset_and_targets(random_state=random_state, M_values=M_values, train_size = 0.2)
            State = Network_State(Variabs)
            State.initiate_resistances(BigClass, R_vec_i)
            State.initiate_accuracy_vec(BigClass, measure_accuracy_every)
            BigClass.add_State(State)  # add to big class
            training_loop()
            # plot_funcs.plot_importants(BigClass, movmean_loss = True, include_network=False, node_labels=False)
            accuracy_in_t.append(State.accuracy_in_t)
            # plot_funcs.plot_accuracy(State.t, State.t_for_accuracy, State.accuracy_in_t, np.shape(Variabs.dataset)[0])
    profiler.disable()
    stats = pstats.Stats(profiler).sort_stats('cumtime')  # or 'tottime'

In [ ]:
loss_mean = statistics.final_err(State.loss_scalar_in_t, samples=400)
print('final loss normalized', loss_mean)
print('STD loss normalized', np.std(State.loss_scalar_in_t[-400:]))

loss_movmean = statistics.mov_ave(State.loss_scalar_in_t, 400)
min_mean_loss = loss_movmean[loss_movmean == np.min(loss_movmean)]
print('min_mean_loss', min_mean_loss)

# Save sizes to file

In [ ]:
save_folder_prelim = 'C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/Network combine/Network_combine/'
# save_folder_prelim = 'C:/Users/roiee/OneDrive - huji.ac.il/PhD/Network Simulation repo/Network combine/Network_combine/'

# # for general regression

# np.save(save_folder_prelim + 't.npy', State.t)
# np.save(save_folder_prelim + 'M.npy', M_values)
# np.save(save_folder_prelim + 'output_lin.npy', np.asarray(State.output_in_t)/np.asarray(State.desired_in_t)-1)
# np.save(save_folder_prelim + 'input_update_lin.npy', State.input_update_in_t)
# np.save(save_folder_prelim + 'output_update_lin.npy', State.output_update_in_t)
# np.save(save_folder_prelim + 'R_lin.npy', State.R_in_t)
# np.save(save_folder_prelim + 'loss_lin.npy', State.loss_in_t)

# import pickle
# # NET.NET networkx graph
# with open(save_folder_prelim + 'NETgraph_4in6out.pkl', 'wb') as f:
#     pickle.dump(NET.NET, f)
    
# # Save the dictionary to a file
# with open(save_folder_prelim + 'pos_lattice_4in6out.pkl', 'wb') as f:
#     pickle.dump(NET.pos_lattice, f)

# # for performance_2_networks
      
# np.save(save_folder_prelim + 'output_4in6out.npy', np.asarray(State.output_in_t))
# np.save(save_folder_prelim + 'input_update_4in6out.npy', State.input_update_in_t)
# np.save(save_folder_prelim + 'output_update_4in6out.npy', State.output_update_in_t)
# np.save(save_folder_prelim + 'R_4in6out_Adalike.npy', State.R_in_t)
# np.save(save_folder_prelim + 'loss_scalar_4in6out_Adalike.npy', State.loss_scalar_in_t)

# import pickle
# # NET.NET networkx graph
# with open(save_folder_prelim + 'NETgraph_1in2out.pkl', 'wb') as f:
#     pickle.dump(NET.NET, f)
    
# # Save the dictionary to a file
# with open(save_folder_prelim + 'pos_lattice_1in2out.pkl', 'wb') as f:
#     pickle.dump(NET.pos_lattice, f)

# np.save(save_folder_prelim + 'cosine_sim_4in6out.npy', cosine_vec)

# for classification_1_material
np.save(save_folder_prelim + 't.npy', State.t)
np.save(save_folder_prelim + 'deltaR_ptopto_Poweraccuracy_in_t.npy', accuracy_in_t)
np.save(save_folder_prelim + 'dataset_shape.npy', np.shape(Variabs.dataset))
np.save(save_folder_prelim + 't_for_accuracy.npy', State.t_for_accuracy)

# for classification_5_material
# np.save(save_folder_prelim + 't.npy', State.t)
# np.save(save_folder_prelim + 'deltaR_propto_dpaccuracy_in_t.npy', accuracy_in_t)
# np.save(save_folder_prelim + 'dataset_shape.npy', np.shape(Variabs.dataset))
# np.save(save_folder_prelim + 't_for_accuracy.npy', State.t_for_accuracy)

# Plots - specific run

## Structure

In [ ]:
# plot_funcs.plotNetStructure(NET.NET, BigClass, NET.pos_lattice, node_labels=False, 
#                                 R_reordered=NET.R_reordered, u_reordered=NET.u_reordered, p_reordered=NET.p_reordered)

## ||Loss|| after movmean

In [ ]:
loss_scalar_moveave = statistics.mov_ave(State.loss_scalar_in_t, 2000)

plt.rcParams['axes.prop_cycle'] = plt.cycler('color', colors_lst)
plt.semilogy(loss_scalar_moveave)
plt.ylabel('$\|\mathcal{L}\|$')
plt.xlabel('$t$')
plt.ylim([2e-3,1]);

## accuracy

In [ ]:
if task_type == 'Iris_classification':
    plot_funcs.plot_accuracy(State.t, State.t_for_accuracy, State.accuracy_in_t, np.shape(Variabs.dataset)[0])
else:
    pass

# Not In Use

## Power consumption

In [ ]:
# if np.size(State.Power_norm_in_t)>1:
#     plot_funcs.plot_Power(State)
# else:
#     pass

## Power of trained network

put a pressure of 1 through all inputs and measure total power dissipation in a trained network that has the state State

In [ ]:
# # Reload the module to reflect any changes made
# importlib.reload(statistics)

# # put pressure of 1 through inputs
# State.input_drawn = np.ones(Nin)

# # solve flow
# State.solve_flow_given_modality(BigClass, "measure")

# # measure power
# print('u', State.u)
# print('Rs', State.R_in_t[-1])
# print('Power dissipation', statistics.power_dissip(State.u, State.R_in_t[-1]))

### R change scheme under 2 tasks

In [ ]:
# import pickle

# load_folder_prelim = 'C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/importants_1in2out_n_2in1out/'
# # load_folder_prelim = 'C:/Users/roiee/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/importants_1in2out_n_2in1out/'
# t = np.load(load_folder_prelim + 't_1in2out.npy')

# loss_1in2out_R_propto_deltap = np.load(load_folder_prelim + 'loss_1in2out_RproptoDeltap.npy')
# loss_1in2out_deltaR_propto_deltap = np.load(load_folder_prelim + 'loss_1in2out_deltaRproptoDeltap.npy')
# loss_1in2out_propto_Q = np.load(load_folder_prelim + 'loss_1in2out_deltaRproptoQ.npy')
# loss_1in2out_propto_Power = np.load(load_folder_prelim + 'loss_1in2out_deltaRproptoPower.npy')
# loss_2in1out_R_propto_deltap = np.load(load_folder_prelim + 'loss_2in1out_RproptoDeltap.npy')
# loss_2in1out_deltaR_propto_deltap = np.load(load_folder_prelim + 'loss_2in1out_deltaRproptoDeltap.npy')
# loss_2in1out_propto_Q = np.load(load_folder_prelim + 'loss_2in1out_deltaRproptoQ.npy')
# loss_2in1out_propto_Power = np.load(load_folder_prelim + 'loss_2in1out_deltaRproptoPower.npy')

# with open(load_folder_prelim + 'NETgraph_1in2out.pkl', 'rb') as f:
#     Network_1in2out = pickle.load(f)
    
# with open(load_folder_prelim + 'NETgraph_2in1out.pkl', 'rb') as f:
#     Network_2in1out = pickle.load(f)
    
# with open(load_folder_prelim + 'pos_lattice_2in1out.pkl', 'rb') as f:
#     pos_lattice = pickle.load(f) 

In [ ]:
# # Reload the module to reflect any changes made
# importlib.reload(figure_plots)

# figure_plots.plot_compare_R_type_loss(Network_1in2out, Network_2in1out, pos_lattice,
#                          loss_1in2out_R_propto_deltap,
#                          loss_1in2out_deltaR_propto_deltap,
#                          loss_1in2out_propto_Q,
#                          loss_1in2out_propto_Power,
#                          loss_2in1out_R_propto_deltap,
#                          loss_2in1out_deltaR_propto_deltap,
#                          loss_2in1out_propto_Q,
#                          loss_2in1out_propto_Power)

## different R relations

In [ ]:
# load_folder_prelim = 'C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/different physical dependence of resistance/'

# # Loading the array later


# R_propto_deltap = np.load(load_folder_prelim + 'R_propto_deltap.npy')
# deltaR_propto_deltap = np.load(load_folder_prelim + '/deltaR_propto_deltap.npy')
# deltaR_propto_Q = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/different physical dependence of resistance/deltaR_propto_Q.npy')
# deltaR_propto_Power = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/different physical dependence of resistance/deltaR_propto_Power.npy')
# loss_R_propto_deltap = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/different physical dependence of resistance/loss_R_propto_deltap.npy')
# loss_deltaR_propto_deltap = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/different physical dependence of resistance/loss_deltaR_propto_deltap.npy')
# loss_propto_Q = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/different physical dependence of resistance/loss_propto_Q.npy')
# loss_propto_Power = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/different physical dependence of resistance/loss_propto_Power.npy')

In [ ]:
# figure_plots.plot_comparison_R_type(R_propto_deltap, deltaR_propto_deltap, deltaR_propto_Q, deltaR_propto_Power,
#                                       loss_R_propto_deltap, loss_deltaR_propto_deltap, loss_propto_Q, loss_propto_Power)

## pseudo vs. network comparison

In [ ]:
# # Loading the array later
# R_in_t_network = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.8/30Aug_R_pseudo_vs_network/R_in_t_network.npy')
# R_in_t_pseudo = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.8/30Aug_R_pseudo_vs_network/R_in_t_pseudo.npy')
# loss_in_t_network = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.8/30Aug_R_pseudo_vs_network/loss_in_t_network.npy')
# loss_in_t_pseudo = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.8/30Aug_R_pseudo_vs_network/loss_in_t_pseudo.npy')

In [ ]:
# figure_plots.plot_comparison_pseudo(R_in_t_pseudo, R_in_t_network, loss_in_t_pseudo, loss_in_t_network)

## comparison Rs GD and Adalike

In [ ]:
# import matplotlib.gridspec as gridspec

# # Set color cycle globally
# plt.rcParams['axes.prop_cycle'] = plt.cycler('color', Colorscheme.colors_lst)

# # Normalize R values so maximal will be 1
# R_GD_norm = R_GD / np.max(R_GD)
# R_ourscheme_norm = R_ourscheme / np.max(R_ourscheme)

# # R_GD_norm = np.concatenate([R_GD_norm[:-1], R_GD_norm[-1:]])
# # R_ourscheme_norm = np.concatenate([R_ourscheme_norm[:-1],  R_ourscheme_norm[-1:]])

# # weird setup for bars
# x = np.arange(len(R_ourscheme_norm))
# bar_width = 0.35

# # instantiate figure and grid for positioning colorbal
# fig = plt.figure(figsize=(7, 3))
# gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], wspace=0.45)

# # R bar plot
# ax0 = fig.add_subplot(gs[0, 0])
# ax0.bar(x - bar_width / 2, R_GD_norm,
#         width=bar_width, label='GD', alpha=0.8, edgecolor='k', linewidth=1.6)
# ax0.bar(x + bar_width / 2, R_ourscheme_norm,
#         width=bar_width, label='this work', alpha=0.8, edgecolor='k', linewidth=1.6)
# ax0.set_xticks([0, 1, 2, 3])
# ax0.set_yticks([0, 0.5, 1])
# ax0.set_ylabel('$R$')
# # ax0.legend()

## Is the network linear?

In [ ]:
# # put pressure of 1 through 1st input
# State.input_drawn = np.array([1,0])

# # solve flow
# State.solve_flow_given_modality(BigClass, "measure")

# # measure power
# print('output 1', State.output)
# out1 = State.output

# # put pressure of 1 through 1st input
# State.input_drawn = np.array([0,1])

# # solve flow
# State.solve_flow_given_modality(BigClass, "measure")

# # measure power
# print('output 2', State.output)
# out2 = State.output

# print('superpose outputs', out1+out2)

# # put pressure of 1 through inputs
# State.input_drawn = np.ones(Nin)

# # solve flow
# State.solve_flow_given_modality(BigClass, "measure")

# # measure power
# print('output both', State.output)

## Most important plot

In [ ]:
importlib.reload(plot_funcs)
if hasattr(Variabs, 'M'):
    plot_funcs.plot_importants(BigClass, movmean_loss = True, include_network=False, node_labels=False)
else:
    plot_funcs.plot_importants(BigClass, include_network=False)

In [ ]:
Strctr.EIEJ_plots[:-6]

## cosine similarity

In [ ]:
if calculate_cosine_sim:
    plt.rcParams['axes.prop_cycle'] = plt.cycler('color', colors_lst)

    loss_movmean = statistics.mov_ave(State.loss_scalar_in_t, 40)

    # Create subplots: 2 rows, 1 column
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=True, figsize=(6, 5))

    # First subplot: resistances
    ax1.plot(BigClass.State.R_in_t[1:])
    ax1.set_ylabel(r'$R$')

    # Second subplot: Cosine similarity
    ax2.set_prop_cycle(plt.cycler('color', colors_lst))  # set color cycle
    ax2.plot(cosine_vec)
    ax2.plot(statistics.mov_ave(cosine_vec, window_size=24))
    ax2.plot(np.zeros([len(cosine_vec)]), '--k')
    ax2.set_ylabel('Cos Similarity')
    ax2.set_ylim([-1, 1])

    # Third subplot: Loss moving average
    ax3.plot(loss_movmean)
    ax3.set_xlabel('$t$')
    ax3.set_ylabel(r'$\|\mathcal{L}\|$')
    ax3.set_yscale('log')
    ax3.set_ylim(1e-4, 1e0)

    plt.tight_layout()
    plt.show()
    
    plt.plot(cosine_vec)
    plt.plot(np.zeros([len(cosine_vec)]), '--k')
    plt.ylabel('Cos Similarity')
    plt.ylim([-1, 1])
    plt.xlabel('$t$')

In [ ]:
np.nanmean(cosine_vec)

In [ ]:
np.mean(cosine_vec[np.isnan(cosine_vec)==False])